[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/25_flash_attention.ipynb)

# 🔴 Hard: Flash Attention (Tiled)

Implement **tiled attention with online softmax** — the core idea behind Flash Attention.

### Signature
```python
def flash_attention(Q, K, V, block_size=32) -> Tensor:
    # Q, K, V: (B, S, D)
    # Returns: (B, S, D) — same as standard attention
```

### Key Insight
Instead of materializing the full S×S attention matrix, process in blocks:
1. For each Q-block, iterate over K/V blocks
2. Use **online softmax**: track running `max` and `sum`
3. Rescale accumulator when max changes: `acc *= exp(old_max - new_max)`
4. Final: `output = acc / row_sum`

Must give **identical** results to standard softmax attention.

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.6 MB/s eta 0:00:00


In [2]:
import torch
import math

In [15]:
# ✏️ YOUR IMPLEMENTATION HERE

def flash_attention(Q, K, V, block_size=32):
    # Process Q in blocks, iterate K/V blocks with online softmax
    shp = Q.shape # (B, S, D)
    B = shp[0]
    S = shp[1]
    D = shp[2]

    num_block = S // block_size

    scores = 0

    m = torch.full((B, S), float('-inf'))
    l = torch.zeros(B, S) #sum
    acc = torch.zeros(B, S, D)

    iter_num = num_block if S % block_size == 0 else num_block + 1
    for i in range(iter_num):
      end = min((i+1)*block_size, S)
      K_tile = K[:, i*block_size : end, :]
      V_tile = V[:, i*block_size : end, :]

      score = torch.einsum("BSD,BTD->BST", Q, K_tile)  / math.sqrt(D)

      # update max
      block_max = torch.max(score, dim=-1).values  #B,S
      m_new = torch.maximum(m, block_max)  #B,S
      # comptue exp
      scale = torch.exp(m - m_new) # B,S

      p = torch.exp(score - m_new.unsqueeze(-1)) # B, S, T
      l = l * scale + p.sum(dim=-1) # B,S
      acc = acc * scale.unsqueeze(-1) + torch.einsum("BST,BTD->BSD", p, V_tile)

      m = m_new

      #score = torch.einsum("BST,BTD->BSD", score, V_tile)
      #scores += score

    return acc / l.unsqueeze(-1)

In [16]:
# 🧪 Debug
import math
Q, K, V = torch.randn(1, 8, 4), torch.randn(1, 8, 4), torch.randn(1, 8, 4)
out = flash_attention(Q, K, V, block_size=4)
scores = torch.bmm(Q, K.transpose(1,2)) / math.sqrt(4)
ref = torch.bmm(torch.softmax(scores, dim=-1), V)
print('Match:', torch.allclose(out, ref, atol=1e-4))

Match: True


In [17]:
# ✅ SUBMIT
from torch_judge import check
check('flash_attention')


🧪 Testing: Flash Attention (Tiled) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Matches standard attention (5.2ms)
  ✅ [2/4] Non-aligned block size (3.9ms)
  ✅ [3/4] Block size invariant (3.3ms)
  ✅ [4/4] Gradient flow (3.3ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (15.6ms total)
  Progress saved. Run status() to see your dashboard.

